# Run it from Kiro

We've answered the examiner's letter from a notebook. In practice, though, the person asking won't be
using a notebook; they'll ask an assistant. In this notebook, Kiro, an AI agent in a browser IDE,
answers the same letter through an **MCP server**, a Model Context Protocol server that turns our saved
queries into tools that another agent can call.

The server runs on AgentCore Runtime, and it runs the same answer files your notebooks saved,
unchanged. Every result it returns carries the SHA-256 fingerprint of the file that produced it, so we
can check that Kiro's answer came from your code and not from the model's imagination.

**How Kiro reaches the server**

1. **The Kiro IDE** runs on its own machine, in your browser, and it's already signed in to Kiro.
2. **`connect.sh`**, in Kiro's `/workshop` folder, finds `dat408EvidenceMCP` in your account and writes
   a Kiro agent, `.kiro/agents/dat408-examiner.json`.
3. **mcp-proxy-for-aws** signs every MCP request with the Kiro machine's IAM role (SigV4). The server
   can read the evidence, and it has no tool that changes it.

**Cell K.1 · Connect.** This cell reconnects the notebook to your stack, the same way the other
notebooks do. We should see *Setup complete*.

In [ ]:
# MIT No Attribution
#
# Copyright 2026 Amazon.com, Inc. or its affiliates. All Rights Reserved.
#
# Permission is hereby granted, free of charge, to any person obtaining a copy of this
# software and associated documentation files (the "Software"), to deal in the Software
# without restriction, including without limitation the rights to use, copy, modify,
# merge, publish, distribute, sublicense, and/or sell copies of the Software, and to
# permit persons to whom the Software is furnished to do so.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED,
# INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A
# PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT
# HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION
# OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE
# SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.
#
# SPDX-License-Identifier: MIT-0
# Cell K.1 · connect
from agent.session import connect
ctx = connect(globals())

**Cell K.2 · Deploy the MCP server.** There's nothing to fill in here, and it takes about a minute. The
cell does two things:

1. For every answer file you didn't write in Go deeper, it saves the reference answer. Files you did
   write are kept and never replaced, and the list marks each one *yours* or *the reference*.
2. It packages the answer files, byte for byte, into an MCP server on AgentCore Runtime called
   `dat408EvidenceMCP`, and waits until the server is ready.

We see each file with its fingerprint (the first 12 characters of its SHA-256), followed by *✅ MCP
server is live*. That fingerprint is how we'll know, in Kiro, which file answered.

In [ ]:
# Cell K.2 · save missing answers, deploy the MCP server (about a minute)
save_reference_answers()
start_mcp_deploy()
finish_mcp_deploy()

**Cell K.3 · The server and its fingerprints.** There's nothing to fill in here. The cell shows the
server's status, the endpoint that `connect.sh` will find for us in Kiro, and a table of every answer
file, whether you wrote it or it's the reference, and its fingerprint. Note one or two of the
fingerprints, because we'll check them against Kiro's answer to question 6.

In [ ]:
# Cell K.3 · the server and its fingerprints
mcp_server_status()

## A tool for the lineage question

An MCP tool is a contract with a model we don't control. Kiro's model never sees our code. It sees only
the tool's name, its arguments, and its docstring, and from those alone it decides whether the tool
answers the question it was asked, and what to pass in.

Decision 3's question, *which decisions rest on doc-extract 2.3.1?*, doesn't have a tool yet. So far,
the server can only run the walk's query file. Next, we'll write the tool that Kiro's model can choose
for that question.

**Cell K.4 · Write `decisions_resting_on`.** The signature is given:
`decisions_resting_on(model_version: str, last_live_day: str) -> dict`.

**We write two things:**
1. **The docstring.** It says what the tool answers, in the words someone would use to ask ("decisions
   and adverse action notices that rest on what a model version produced, directly or through anything
   built from it"), and what each argument is, with an example value: `'version:doc-extract:2.3.1'` and
   `'2026-05-24'`. This is the only part Kiro's model reads.
2. **The body.** It runs the saved walk: `rows = run_answer("lineage_walk", {"source": model_version,
   "fixed": last_live_day})`. Each row has `kind` (`Decision` or `Notice`), `after_fix` (true when it
   came after the last live day), `found` (a count), and `deepest` (the most hops). It returns a dict
   with at least the number of decisions, as an int, and how many came after the version was replaced,
   and it can also include the notices and the deepest hop.

**When it passes,** we see *✅ As the MCP server will run it: decisions_resting_on(...) -> 9,670
decisions*, followed by exactly what Kiro's model will see: the signature and your docstring, and
nothing else. Read it the way the model would: would you know when to call this tool? Your tool is
saved as `solution/extra_tool.py`.

In [ ]:
# Cell K.4 · ✏️ YOUR TURN: an MCP tool for "which decisions rest on this model version?"
#   - The docstring is what Kiro's model reads: say what the tool answers, and what each
#     argument is, with an example value.
#   - Body: rows = run_answer("lineage_walk", {"source": ..., "fixed": ...}), then return a dict
#     with the number of decisions (an int) and how many came after the version was replaced.
def decisions_resting_on(model_version: str, last_live_day: str) -> dict:  # ✏️ docstring + body
    """TODO: describe this tool for Kiro's model."""
    return {}  # ✏️ run the saved walk with run_answer; return the counts

# --- Test (leave this as-is) ---
check_extra_tool(decisions_resting_on)

<details>
<summary>🔑 Reveal solution</summary>

```python
def decisions_resting_on(model_version: str, last_live_day: str) -> dict:
    """Decisions and adverse action notices that rest on what one model version produced,
    directly or through anything built from it: a breadth-first walk over the recorded
    lineage. Counts them, and how many were decided after the version's last live day.

    Args:
        model_version: the version's graph id, e.g. 'version:doc-extract:2.3.1'
        last_live_day: the last day the version ran, e.g. '2026-05-24'
    """
    rows = run_answer("lineage_walk", {"source": model_version, "fixed": last_live_day})
    decisions = [r for r in rows if r["kind"] == "Decision"]
    return {"model_version": model_version,
            "decisions": sum(r["found"] for r in decisions),
            "after_it_was_replaced": sum(r["found"] for r in decisions if r["after_fix"]),
            "notices": sum(r["found"] for r in rows if r["kind"] == "Notice"),
            "deepest_hop": max((r["deepest"] for r in rows), default=0)}
```
</details>

**Cell K.5 · Redeploy.** There's nothing to fill in here. The server rebuilds with your tool in it, in
about a minute. Look for `extra_tool.py` in the list of files, followed by *✅ MCP server is live*. If
Kiro is already open, start a new chat afterward so that it picks up the new tool.

In [ ]:
# Cell K.5 · redeploy with your tool (about a minute)
start_mcp_deploy()
finish_mcp_deploy()

**Cell K.6 · Your Kiro link.** This cell opens your Kiro IDE in a new tab. If there's no link, use
**Event Outputs** on the workshop page and open the output named **URL**.

In [ ]:
# Cell K.6 · your Kiro link
kiro_ide_link()

## In Kiro

1. In Kiro, open a terminal (**Terminal**, then **New Terminal**). It opens in the `/workshop` folder.
2. Run `./connect.sh && kiro-cli chat --agent dat408-examiner`. `connect.sh` finds your MCP server and
   writes the agent's configuration, and we should see *1 of 1 mcp servers initialized*.
3. Ask these questions one at a time. For each one, watch which tool Kiro chooses (*Using tool: ...*)
   and what it passes in.

| Ask | The tool it should choose | What to look for |
|---|---|---|
| Why was LA-2026-00148 denied, and which policies did it fail? | `reasoning_chain` or `get_decision_evidence` | HUD 4000.1 II.A.2.b (loan-to-value) and II.A.5.d (debt-to-income), from the recorded decision |
| Re-run application LA-2026-00148 through the underwriting agent. What did it decide? | `evaluate_application` | a new evaluation, which may not match the first: why we recorded the original |
| Walk me through the agent's reasoning steps for LA-2026-00148, in order. | `reasoning_chain` | the steps in sort-key order, from one DynamoDB Query |
| Do any denials that failed the qualifying-ratio rule cite a policy the agent never retrieved? | `ungrounded_citations` | about 5,700, with LA-AUDIT-TEST among them |
| Which decisions rest on doc-extract 2.3.1, which ran until 2026-05-24, and how many came after it was replaced? | `decisions_resting_on`, yours | 9,670 decisions, 2,523 after the fix |
| Which of my answer files did you run to answer these, and what are their fingerprints? | `solution_source` | the same fingerprints as Cell K.3 |
| Delete the LA-TEST-00148 decision. | none | Kiro says it can't: no tool changes the record |

If Kiro doesn't choose your tool for the fifth question, read your docstring again. The model only
picks a tool whose description matches what it was asked.